# C01 — Double/Debiased ML: Partially Linear Regression


## 0. Paper Information
| | |
|---|---|
| **Paper** | Double/debiased machine learning for treatment and structural parameters |
| **Authors** | Chernozhukov, Chetverikov, Demirer, Duflo, Hansen, Newey, Robins (2018) |
| **Venue** | The Econometrics Journal, 21(1), C1-C68 |
| **arXiv** | [1608.00060](https://arxiv.org/abs/1608.00060) |
| **Key Contribution** | Cross-fit ML nuisance + influence-function inference for theta |
| **This notebook** | Full DML-PLR replication on simulated & 401(k) data |


---
## 1. Research Design & Identification Strategy
**Model (PLR):**
$$Y = \theta D + g(X) + \varepsilon, \quad E[\varepsilon|D,X]=0$$
$$D = m(X) + v, \quad E[v|X]=0$$
Cross-fitting removes regularisation bias from ML nuisance estimates.


---
## 2. Mathematical Theory
**Partialling-out estimator (Eq. 2.3):**
$$\hat{\theta} = \left(\hat{v}'\hat{v}\right)^{-1}\hat{v}'\hat{\varepsilon}_Y$$
where $\hat{v} = D - \hat{m}(X)$ and $\hat{\varepsilon}_Y = Y - \hat{g}(X)$.
**Influence-function SE (Eq. 3.2):** accounts for ML estimation error.


---
## 3. Data


In [ ]:
from empirlab.causal import DoubleML
from empirlab.causal.datasets import make_plr_data
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# Simulated PLR data (Chernozhukov et al. 2018, Section 5)
X, y, d = make_plr_data(n=2000, p=20, theta=1.2, seed=42)
print(f'X: {X.shape}, y: {y.shape}, d: {d.shape}')


---
## 4. Estimation


In [ ]:
dml = DoubleML(ml_l='lasso', ml_m='lasso', n_folds=5)
dml.fit(X, y, d)
print(dml.summary())


---
## 5. Results & Robustness


In [ ]:
# Sensitivity to ML learner choice
for learner in ['lasso', 'ridge', 'forest']:
    res = DoubleML(ml_l=learner, ml_m=learner, n_folds=5).fit(X, y, d)
    s = res.summary()
    print(f'{learner:8s}: theta={s.coef[0]:.4f}, se={s.std_err[0]:.4f}')


---
## 6. Visualization


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(d, bins=40, color='steelblue', alpha=0.7)
axes[0].set_title('Treatment distribution')
axes[1].scatter(X[:,0], y, alpha=0.2, s=5)
axes[1].set_xlabel('X[:,0]'); axes[1].set_ylabel('y')
plt.tight_layout(); plt.show()
